# DNS feature selection: repeatable runs

This notebook repeats the **final evaluation** of the feature subsets you already selected, using several random seeds, and (optionally) adds a random-search baseline and a controller audit.

**What it does and does not test**

| Part | Repeats | What it tells you |
|---|---|---|
| A | Final Random Forest / XGBoost fits and test predictions, subsets fixed, different seeds | Whether F1, error counts and runtimes are stable against classifier randomness |
| B1 | Quick Enhanced RLGWO controller audit (small data, minutes) | Whether the Dueling DQN receives any network updates in 15 generations |
| B2 | Random-search baseline, with the same fitness function and a stated budget | Whether a simple search finds subsets as good as ACO, PSO or Enhanced RLGWO |
| B3 | Full optimiser searches with extra seeds (off by default, hours each) | Stability of the searches themselves |

**Files needed beside this notebook:** `dns_eval_utils.py`, `dns_search_utils.py`, `x_train.npy`, `y_train.npy`, `x_val.npy`, `y_val.npy`, `x_test.npy`, `y_test.npy`, the earlier results folder `validation_only_run_v1/`, and (Parts B1/B3) your `Enhanced_detection.py`.

**Rules to keep the results honest:** choose the seeds before you look at results; do not drop or favour a seed; never select a subset using test results; report every run you make.

Results are saved one file per run, so you can stop and restart. Finished runs are skipped.

In [ ]:
# 1. CONFIGURATION: edit this cell only
from pathlib import Path

SAVED_RUN_DIR = Path("validation_only_run_v1")   # your earlier corrected run
RUN_DIR       = Path("repeat_runs_v1")           # new folder for repeat results

SEEDS            = [42, 43, 44]   # decide BEFORE looking at results; 42 reproduces the original run
N_PRED_REPEATS   = 5              # repeated test predictions per fitted model (median reported)
SAVE_PREDICTIONS = False          # True saves one uint8 prediction file per run (~7 MB each)

# Optional parts (all off by default)
RUN_CONTROLLER_AUDIT = False      # B1: minutes; needs Enhanced_detection.py
RUN_RANDOM_SEARCH    = False      # B2: needs the validation search samples below
RANDOM_SEARCH_EVALS  = 600        # ACO made 40 x 15 = 600 fitness calls; PSO made 640
RANDOM_SEARCH_SEEDS  = [42]
RUN_FULL_SEARCHES    = False      # B3: hours per seed per method
SEARCH_SEEDS         = [43]
SEARCH_METHODS       = ["ACO", "PSO", "Enhanced RLGWO"]

N_SEARCH_TRAIN, N_SEARCH_VAL = 500_000, 150_000
POP_SIZE, GENERATIONS = 40, 15
FEATURE_NAMES = ["len","subdomains_count","w_count","w_max","entropy","w_max_ratio","w_count_ratio",
                 "digits_ratio","uppercase_ratio","time_avg","time_stdev","size_avg","size_stdev",
                 "throughput","unique","entropy_avg","entropy_stdev"]
RUN_DIR.mkdir(exist_ok=True)
(RUN_DIR / "figures").mkdir(exist_ok=True)
print("Seeds:", SEEDS, "| results folder:", RUN_DIR)

In [ ]:
# 2. Imports and data (arrays are memory-mapped, not duplicated)
import json, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dns_eval_utils as eu
import dns_search_utils as su

arrays = {}
for name in ["x_train", "y_train", "x_val", "y_val", "x_test", "y_test"]:
    path = Path(f"{name}.npy")
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run your original extraction/cache step first.")
    arrays[name] = np.load(path, mmap_mode="r", allow_pickle=False)

for split in ["train", "val", "test"]:
    X, y = arrays[f"x_{split}"], arrays[f"y_{split}"]
    assert X.shape[1] == 17 and len(X) == len(y)
    print(split, X.shape, dict(zip(*np.unique(y, return_counts=True))))

In [ ]:
# 3. Load the feature subsets selected in your corrected run
def slug(text): return text.lower().replace(" ", "_")

configs = {"Baseline": list(range(17))}
for name in ["ACO", "PSO", "Enhanced RLGWO"]:
    record = json.loads((SAVED_RUN_DIR / f"{slug(name)}_search.json").read_text())
    assert record.get("search_evaluation_partition") == "validation", f"{name}: not a validation-only search"
    configs[name] = record["selected_indices"]

for name, idx in configs.items():
    print(f"{name:15s} {len(idx):2d} features: {idx}")

## Part A: repeat the final evaluation with different seeds

Subsets are fixed. Only the classifier random seed changes (Random Forest sampling, XGBoost row subsampling). Training time covers `fit()` only. Prediction is repeated and the **median** is stored.

Rough cost: from your recorded times, one pass over all eight models is about 40 minutes of fitting, so three seeds is about two hours plus prediction and loading.

In [ ]:
# 4. Run the repeats (resumable: finished runs are skipped)
total = len(SEEDS) * len(configs) * 2
done = 0
for seed in SEEDS:
    for config, idx in configs.items():
        for key in ["xgb", "rf"]:
            started = eu.result_path(RUN_DIR, config, key, seed).exists()
            r = eu.evaluate_one(config, idx, key, seed, arrays, RUN_DIR,
                                n_pred_repeats=N_PRED_REPEATS, save_predictions=SAVE_PREDICTIONS)
            done += 1
            tag = "loaded" if started else "ran   "
            print(f"[{done}/{total}] {tag} seed={seed} {config:15s} {key:3s} "
                  f"F1={r['f1']:.6f} errors={r['total_errors']} "
                  f"train={r['train_seconds']:.1f}s predict(median)={r['inference_median']:.3f}s")

In [ ]:
# 5. Reproducibility check: does seed 42 match your original run?
# Counts should match closely. Times will differ, which is why repeats are useful.
if 42 in SEEDS:
    rows = []
    for config in configs:
        for key in ["xgb", "rf"]:
            old_path = SAVED_RUN_DIR / f"{slug(config)}_{key}_cm.npy"
            if not old_path.exists():
                continue
            old = np.load(old_path).ravel().tolist()               # tn, fp, fn, tp
            new_r = json.loads(eu.result_path(RUN_DIR, config, key, 42).read_text())
            new = [new_r["tn"], new_r["fp"], new_r["fn"], new_r["tp"]]
            rows.append(dict(config=config, model=key, original=old, repeat=new, identical=old == new))
    check = pd.DataFrame(rows)
    display(check)
    print("All identical:", bool(check.identical.all()) if len(check) else "no saved matrices found")

In [ ]:
# 6. Summary across seeds
df = eu.load_all(RUN_DIR)
order = list(configs) + [c for c in df.config.unique() if c not in configs]
summary = eu.summarise(df, baseline="Baseline", order=order)
summary.to_csv(RUN_DIR / "summary_across_seeds.csv", index=False)

show = ["config", "classifier", "n_seeds", "f1_mean", "f1_sd", "total_errors_mean", "total_errors_sd",
        "train_seconds_mean", "train_seconds_reduction_pct", "inference_median_mean", "inference_median_reduction_pct"]
display(summary[show].round(6))
df.to_csv(RUN_DIR / "all_runs.csv", index=False)

In [ ]:
# 7. Is each ranking stable across seeds? (per-seed difference in F1, A minus B)
pairs = [("Enhanced RLGWO", "Baseline"), ("Enhanced RLGWO", "PSO"), ("Enhanced RLGWO", "ACO"), ("PSO", "Baseline")]
rows = []
for key in ["xgb", "rf"]:
    for a, b in pairs:
        d = eu.paired_seed_differences(df, a, b, key, "f1")
        if len(d):
            rows.append(dict(classifier=key, comparison=f"{a} - {b}", seeds=len(d),
                             mean_diff=d.mean(), min_diff=d.min(), max_diff=d.max(),
                             same_sign_every_seed=bool((d > 0).all() or (d < 0).all())))
stability = pd.DataFrame(rows)
stability.to_csv(RUN_DIR / "ranking_stability.csv", index=False)
display(stability.round(6))

In [ ]:
# 8. Figures and LaTeX table (no colour-bar caption text)
colors = {"xgb": "#4472C4", "rf": "#ED7D31"}
plt.rcParams.update({"font.family": "serif", "font.size": 10, "pdf.fonttype": 42})
names = [c for c in order if c in set(summary.config)]
x = np.arange(len(names)); w = 0.36

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (col, sd, label) in zip(axes, [("f1_mean", "f1_sd", "Attack-class F1 (truncated axis)"),
                                       ("train_seconds_mean", "train_seconds_sd", "Training time (s)"),
                                       ("inference_median_mean", "inference_median_sd", "Batch prediction time (s)")]):
    for off, key in [(-w/2, "xgb"), (w/2, "rf")]:
        s = summary[summary.classifier == key].set_index("config").loc[names]
        ax.bar(x + off, s[col], w, yerr=s[sd].fillna(0), capsize=3, color=colors[key], label=key.upper())
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=20); ax.set_ylabel(label)
    if col == "f1_mean":
        ax.set_ylim(max(0, summary.f1_mean.min() - 0.003), 1.002)
    ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(RUN_DIR / "figures" / "repeat_summary.pdf", bbox_inches="tight")
fig.savefig(RUN_DIR / "figures" / "repeat_summary.png", dpi=300, bbox_inches="tight")
plt.show()

latex = eu.latex_summary_table(summary, f"Mean and standard deviation over {len(SEEDS)} classifier seeds "
                               "with fixed feature subsets. Prediction time is the median of repeated "
                               "batch predictions.", "tab:repeat-summary")
(RUN_DIR / "repeat_summary_table.tex").write_text(latex)
print(latex)

## Part B1: Enhanced RLGWO controller audit (small and fast)

Whether the Dueling DQN trains depends on the number of generations and the replay batch size, not on the data size. This cell therefore uses **small** random samples and a tiny population so it finishes in minutes, but keeps your 15 generations. It counts calls to the network optimiser's `step()`.

Interpretation:
- `network_updates == 0`: the controller's network was never updated in 15 generations. Report that, and do not describe the run as demonstrating learned control.
- `network_updates > 0`: record the number and the action counts as evidence.
- `tracked == False`: your module does not expose `optimizer`. Open `Enhanced_detection.py` and check the replay code by hand.

In [ ]:
# 9. B1: controller audit (set RUN_CONTROLLER_AUDIT = True in the config cell)
import importlib.util
if RUN_CONTROLLER_AUDIT:
    module_path = Path("Enhanced_detection.py")
    assert module_path.exists(), "Place your original Enhanced_detection.py beside this notebook."
    spec = importlib.util.spec_from_file_location("enhanced_detection", module_path)
    enh = importlib.util.module_from_spec(spec); spec.loader.exec_module(enh)

    rng = np.random.default_rng(0)
    tr = np.sort(rng.choice(len(arrays["y_train"]), 20_000, replace=False))
    va = np.sort(rng.choice(len(arrays["y_val"]), 10_000, replace=False))
    Xa, ya = np.asarray(arrays["x_train"][tr]), np.asarray(arrays["y_train"][tr])
    Xb, yb = np.asarray(arrays["x_val"][va]), np.asarray(arrays["y_val"][va])
    enh.X_train_all, enh.y_train_all, enh.X_test_all, enh.y_test_all = Xa, ya, Xb, yb

    fit_audit, stats_audit = su.make_fitness(Xa, ya, Xb, yb, seed=42, n_trees=10)
    su.seed_everything(42)
    controller = enh.EnhancedBinaryRLGWO(17, 6, GENERATIONS)
    counter = su.attach_update_counter(controller)
    controller.run(fit_audit)
    audit = dict(generations=GENERATIONS, population=6, controller_updates=counter,
                 controller_info=su.describe_controller(controller), fitness_calls=stats_audit["calls"])
    (RUN_DIR / "controller_audit.json").write_text(json.dumps(audit, indent=2))
    print(json.dumps(audit, indent=2))
else:
    print("Controller audit skipped.")

## Part B2: random-search baseline

Uses the **same** fitness function (random forest with 50 trees, fitted on 500,000 training records, scored on 150,000 validation records) and the same 8 to 10 feature limit. The budget is stated in fitness calls. Only the subset with the best **validation** fitness is evaluated on the test set, once per seed.

This is a budget-matched comparison in fitness calls only. Report `fitness_calls` and `rf_fits` from the saved search record, because ACO, PSO and Enhanced RLGWO do not all train the same number of forests.

In [ ]:
# 10. Validation search samples and fitness (needed by B2 and B3)
def build_fitness(seed=42):
    from sklearn.model_selection import train_test_split
    y_tr, y_va = np.asarray(arrays["y_train"]), np.asarray(arrays["y_val"])
    tr_idx, _ = train_test_split(np.arange(len(y_tr)), train_size=N_SEARCH_TRAIN, stratify=y_tr, random_state=seed)
    va_idx, _ = train_test_split(np.arange(len(y_va)), train_size=N_SEARCH_VAL, stratify=y_va, random_state=seed)
    Xs, ys = np.asarray(arrays["x_train"][np.sort(tr_idx)]), y_tr[np.sort(tr_idx)]
    Xv, yv = np.asarray(arrays["x_val"][np.sort(va_idx)]), y_va[np.sort(va_idx)]
    fit, stats = su.make_fitness(Xs, ys, Xv, yv, seed=seed, n_jobs=1, n_trees=50)
    return fit, stats, (Xs, ys, Xv, yv)

if RUN_RANDOM_SEARCH or RUN_FULL_SEARCHES:
    fitness, search_stats, search_data = build_fitness(42)
    print("Search samples:", search_data[0].shape, search_data[2].shape, "(validation only)")
else:
    print("Search samples not built (no search part enabled).")

In [ ]:
# 11. B2: random search, then final evaluation of its selected subset
random_configs = {}
if RUN_RANDOM_SEARCH:
    for seed in RANDOM_SEARCH_SEEDS:
        rec = su.run_search(f"Random search s{seed}", lambda: su.RandomSearch(17, RANDOM_SEARCH_EVALS),
                            fitness, search_stats, seed, RUN_DIR / "searches")
        name = f"Random search s{seed}"
        random_configs[name] = rec["selected_indices"]
        for key in ["xgb", "rf"]:
            for eval_seed in SEEDS:
                eu.evaluate_one(name, rec["selected_indices"], key, eval_seed, arrays, RUN_DIR,
                                n_pred_repeats=N_PRED_REPEATS)
    print("Random-search subsets:", random_configs)
else:
    print("Random search skipped.")

## Part B3: repeat the full optimiser searches (optional, slow)

From your earlier run: ACO about 1 hour, PSO about 2 h 53 min, Enhanced RLGWO about 2 h 57 min per seed. Run one method and one seed at a time, and keep the notebook open. Each finished search is saved. The subset found is then evaluated with the same final models.

Do not pick the best seed after seeing test results. Report all of them.

In [ ]:
# 12. B3: full searches with extra seeds (set RUN_FULL_SEARCHES = True)
extra_configs = {}
if RUN_FULL_SEARCHES:
    spec = importlib.util.spec_from_file_location("enhanced_detection", Path("Enhanced_detection.py"))
    enh = importlib.util.module_from_spec(spec); spec.loader.exec_module(enh)
    Xs, ys, Xv, yv = search_data
    enh.X_train_all, enh.y_train_all, enh.X_test_all, enh.y_test_all = Xs, ys, Xv, yv   # validation data, not test

    factories = {"ACO": lambda: su.BinaryACO(17, POP_SIZE, GENERATIONS),
                 "PSO": lambda: su.BinaryPSO(17, POP_SIZE, GENERATIONS),
                 "Enhanced RLGWO": lambda: enh.EnhancedBinaryRLGWO(17, POP_SIZE, GENERATIONS)}
    for seed in SEARCH_SEEDS:
        for method in SEARCH_METHODS:
            rec = su.run_search(method, factories[method], fitness, search_stats, seed, RUN_DIR / "searches")
            name = f"{method} s{seed}"
            extra_configs[name] = rec["selected_indices"]
            for key in ["xgb", "rf"]:
                for eval_seed in SEEDS:
                    eu.evaluate_one(name, rec["selected_indices"], key, eval_seed, arrays, RUN_DIR,
                                    n_pred_repeats=N_PRED_REPEATS)
    print(extra_configs)
else:
    print("Full searches skipped.")

In [ ]:
# 13. Which features each search picked, across everything run so far
rows = []
for path in sorted((RUN_DIR / "searches").glob("*_search.json")) if (RUN_DIR / "searches").exists() else []:
    r = json.loads(path.read_text())
    rows.append(dict(method=r["name"], seed=r["seed"], k=len(r["selected_indices"]),
                     validation_fitness=r["best_validation_fitness"], fitness_calls=r["fitness_calls"],
                     rf_fits=r["rf_fits"], search_hours=r["search_seconds"]/3600,
                     features=", ".join(FEATURE_NAMES[i] for i in r["selected_indices"])))
searches = pd.DataFrame(rows)
if len(searches):
    searches.to_csv(RUN_DIR / "search_summary.csv", index=False)
    display(searches.round(5))
else:
    print("No new searches recorded yet.")

## How to report these results

- Say how many seeds you used and that they were fixed in advance.
- Report mean and standard deviation (or all values). With three seeds a standard deviation is only a rough guide.
- Part A repeats the **classifier** randomness with fixed subsets. It does not show how stable the optimisers are. Only B3 does.
- If the controller audit shows zero network updates, say so in the methodology and limitations.
- A random-search result that matches or beats an optimiser is a valid finding. Report it as it is.
- These repeats use the same data partitions. They do not address overlapping request windows or earlier inspection of the test set.